# setup

In [1]:
import rclpy
from rclpy.node import Node
from geometry_msgs.msg import Pose
from sensor_msgs.msg import JointState

from pymoveit2 import MoveIt2
from tf2_ros import Buffer, TransformListener

import json
import time

In [2]:
rclpy.init()

node = Node("behavior_recorder_jupyter_node1")

print("ROS initialized")

ROS initialized


In [3]:
import time

def spin_some(duration=2.0):
    """    
    important: take some time to process incoming data!
    """
    import time
    end = time.time() + duration
    while time.time() < end:
        rclpy.spin_once(node, timeout_sec=0.1)

In [4]:
from pymoveit2 import MoveIt2
from tf2_ros import Buffer, TransformListener

from geometry_msgs.msg import Pose

tf_buffer = Buffer()
tf_listener = TransformListener(tf_buffer, node)

spin_some() # give it some time to update

In [5]:
moveit2 = MoveIt2(
    node=node,
    joint_names=[
        "joint_1","joint_2","joint_3",
        "joint_4","joint_5","joint_6",
    ],
    base_link_name="base_link",
    end_effector_name="tool_frame",
    group_name="arm",
)

# gripper

In [6]:
from rclpy.action import ActionClient
from control_msgs.action import GripperCommand

gripper_client = ActionClient(
    node,
    GripperCommand,
    "/gen3_lite_2f_gripper_controller/gripper_cmd"
)

In [7]:
def control_gripper(position, effort=1.0):
    """
    position:
        0.0 = open
        ~0.8–1.0 = closed
    """

    goal_msg = GripperCommand.Goal()
    goal_msg.command.position = position
    goal_msg.command.max_effort = effort

    print(f"Sending gripper command: {position}")

    # wait for server
    gripper_client.wait_for_server()

    send_goal_future = gripper_client.send_goal_async(goal_msg)

    rclpy.spin_until_future_complete(node, send_goal_future)
    goal_handle = send_goal_future.result()

    if not goal_handle.accepted:
        print("Gripper goal rejected")
        return

    result_future = goal_handle.get_result_async()
    rclpy.spin_until_future_complete(node, result_future)

    print("Gripper done")

def open_gripper():
    control_gripper(0.0)

def close_gripper():
    control_gripper(0.8)  # adjust if needed

In [8]:
close_gripper()
time.sleep(1)
open_gripper()

Sending gripper command: 0.8
Gripper done
Sending gripper command: 0.0
Gripper done


# get pose

In [9]:
def get_pose():
    spin_some(1.0)
    try:
        trans = tf_buffer.lookup_transform(
            "base_link",
            "tool_frame",
            rclpy.time.Time()
        )

        pose = Pose()
        pose.position.x = trans.transform.translation.x
        pose.position.y = trans.transform.translation.y
        pose.position.z = trans.transform.translation.z
        pose.orientation = trans.transform.rotation

        return pose
    except Exception as e:
        print("TF error:", e)
        return None
    
def print_pose(pose):
    print("Translation:",
          [pose.position.x, pose.position.y, pose.position.z])
    print("Quaternion:",
          [pose.orientation.x,
           pose.orientation.y,
           pose.orientation.z,
           pose.orientation.w])

test_pose0 = get_pose()
if test_pose0:
    print("Current pose:")
    print_pose(test_pose0)
else:
    print("failed to get pose.")
    

def make_pose(pos_xyz, orient_xyzw):
    pose = Pose()
    
    pose.position.x, pose.position.y, pose.position.z = pos_xyz
    pose.orientation.x, pose.orientation.y, pose.orientation.z, pose.orientation.w = orient_xyzw
    
    return pose

Current pose:
Translation: [0.23977524464693692, -0.11653249590215686, 0.19856424229050823]
Quaternion: [-0.6346024025190221, -0.32016492888516607, 0.6799283941227602, -0.18019874554137635]


## save pose

In [10]:
grey_bottle_grasp_pose = make_pose(
    pos_xyz=(
        0.5037365889870258, 
        -0.2240841929310071, 
        0.11609242244597344
        ), 
    orient_xyzw=(
        0.7301995308282448, 
        -0.005421623908711407, 
        0.033587134403944244, 
        0.6823863682511068
        )
    )
print_pose(grey_bottle_grasp_pose)

Translation: [0.5037365889870258, -0.2240841929310071, 0.11609242244597344]
Quaternion: [0.7301995308282448, -0.005421623908711407, 0.033587134403944244, 0.6823863682511068]


In [11]:
# grey_bottle_pre_grasp_pose = get_pose()
# print_pose(grey_bottle_pre_grasp_pose)

In [12]:
grey_bottle_pre_grasp_pose =  make_pose(
    pos_xyz=[0.5035170876022704, -0.12519245872336004, 0.13016054581559786],
    orient_xyzw=[0.7122703498240701, 0.018009366124129053, 
                 0.044352911487721705, 0.700270969508137]
)
print_pose(grey_bottle_pre_grasp_pose)

Translation: [0.5035170876022704, -0.12519245872336004, 0.13016054581559786]
Quaternion: [0.7122703498240701, 0.018009366124129053, 0.044352911487721705, 0.700270969508137]


In [13]:
above_mircophone_pose = make_pose(
    pos_xyz=[
        0.5035170876022704,
        -0.12519245872336004,
        0.13016054581559786 + 0.20
    ],
    orient_xyzw=[
        0.7122703498240701,
        0.018009366124129053,
        0.044352911487721705,
        0.700270969508137
    ]
)

print_pose(above_mircophone_pose)

Translation: [0.5035170876022704, -0.12519245872336004, 0.33016054581559784]
Quaternion: [0.7122703498240701, 0.018009366124129053, 0.044352911487721705, 0.700270969508137]


In [14]:
# cup_pen_pre_grasp_pose = get_pose()
# print_pose(cup_pen_pre_grasp_pose)

cup_pen_pre_grasp_pose = make_pose(
    pos_xyz=[0.35346299290800076, -0.12890278627088447, 0.04871254200241307],
    orient_xyzw=[0.8039581741893985, 0.026671357129026064, 0.024053949195080093, 0.5936002867174723]
)


In [15]:
# cup_pen_grasp_pose = get_pose()
# print_pose(cup_pen_grasp_pose)

cup_pen_grasp_pose = make_pose(
    pos_xyz=[0.3470795335833769, -0.17505817482321506, 0.048968048074364764],
    orient_xyzw=[0.7999772075325435, -0.0059638605729074095, 0.00583137462724921, 0.5999724117536216]
)

In [16]:
# cup_tilt_pose = get_pose()
# print_pose(cup_tilt_pose)

cup_tilt_pose = make_pose(
    pos_xyz=[0.28762861267741296, -0.1889354706791522, 0.20975108669772796],
    orient_xyzw=[0.9729684284898917, 0.1697256380619787, 0.03949216785938783, -0.1515454176942695]
)

In [17]:
# cup_pre_tilt_pose = get_pose()
# print_pose(cup_pre_tilt_pose)

cup_pre_tilt_pose = make_pose(
    pos_xyz=[0.34769987911092504, -0.1751015310279324, 0.24681058864261823],
    orient_xyzw=[0.8005613986661017, -0.005208209991114569, 0.00414769912108677, 0.5992137499310779]
)

# get joint state

In [18]:
joint_state = None

def joint_callback(msg):
    global joint_state
    joint_state = msg

node.create_subscription(
    JointState,
    "/joint_states",
    joint_callback,
    10
)

spin_some(1)

print(joint_state)

sensor_msgs.msg.JointState(header=std_msgs.msg.Header(stamp=builtin_interfaces.msg.Time(sec=1777659539, nanosec=202601945), frame_id='base_link'), name=['right_finger_bottom_joint', 'joint_1', 'joint_2', 'joint_4', 'joint_5', 'joint_3', 'joint_6'], position=[-0.02520232172012329, 0.03898867835876005, -1.0573836097339333, 0.6239120643783136, 2.244204837928987, 1.1319574465783315, -0.9306714703323982], velocity=[0.0, 2.3776571849971927e-05, -5.811993508731449e-05, -0.00011102011344922246, 3.7006337399000555e-05, -2.3790024600809194e-05, 0.00011095733800897656], effort=[nan, -0.4467518925666809, -9.84866714477539, 0.29951921105384827, -0.579070508480072, 2.3231098651885986, 0.2961912453174591])


# move pose

In [19]:
DEFAULT_SPEED = 0.3

import numpy as np

def move_to_pose(p, speed=None, wait=3.0):


    # Save current settings
    prev_vel = moveit2.max_velocity
    prev_acc = moveit2.max_acceleration

    # Apply new speed (or default)
    if speed is None:
        speed = DEFAULT_SPEED

    moveit2.max_velocity = speed
    moveit2.max_acceleration = speed

    # Execute motion
    moveit2.move_to_pose(p)

    spin_some(wait)   # assuming background spin

    # Restore previous settings
    moveit2.max_velocity = prev_vel
    moveit2.max_acceleration = prev_acc

    return None

def reached_pose(target, current=None, pos_tol=0.01, ori_tol=0.05, check_orient=True):
    
    if current is None:
        current = get_pose()
        
    # position
    p1 = np.array([
        current.position.x,
        current.position.y,
        current.position.z
    ])

    p2 = np.array([
        target.position.x,
        target.position.y,
        target.position.z
    ])

    pos_error = np.linalg.norm(p1 - p2)

    if not check_orient:
        return pos_error < pos_tol

    else:
        # orientation (quaternion distance)
        q1 = np.array([
            current.orientation.x,
            current.orientation.y,
            current.orientation.z,
            current.orientation.w
        ])

        q2 = np.array([
            target.orientation.x,
            target.orientation.y,
            target.orientation.z,
            target.orientation.w
        ])

        ori_error = np.linalg.norm(q1 - q2)

        return pos_error < pos_tol and ori_error < ori_tol


In [ ]:
def sync_state():
    time.sleep(0.5)          # let controller settle
    spin_some(0.5)           # update TF + joints

def wait_until_stable(timeout=5.0, tol=1e-3):
    import numpy as np
    prev = None
    start = time.time()

    while time.time() - start < timeout:
        spin_some(0.2)
        p = get_pose()

        curr = np.array([
            p.position.x,
            p.position.y,
            p.position.z
        ])

        if prev is not None:
            if np.linalg.norm(curr - prev) < tol:
                return True

        prev = curr

    return False

def move_to_pose_robust(p, speed=None):
    
    # --- sync ---
    sync_state()

    # --- speed ---
    if speed is None:
        speed = DEFAULT_SPEED

    moveit2.max_velocity = speed
    moveit2.max_acceleration = speed

    # --- plan ---
    plan = moveit2.plan(pose=p)

    if plan is None:
        print("❌ Planning failed")
        return False

    # --- execute ---
    moveit2.execute(plan)

    # --- wait for completion ---
    wait_until_stable()

    # --- verify ---
    if not reached_pose(p):
        print("⚠️ Execution finished but pose not reached")
        return False

    return True

In [21]:
move_to_pose_robust(grey_bottle_pre_grasp_pose, speed=0.2)
move_to_pose_robust(grey_bottle_grasp_pose, speed=0.2)

[WARN] [1777659547.929413595] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777659547.930905353] [behavior_recorder_jupyter_node1]: Joint states are available now
[WARN] [1777659548.966782907] [behavior_recorder_jupyter_node1]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.


⚠️ Execution finished but pose not reached


[WARN] [1777659552.387073476] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777659552.387996300] [behavior_recorder_jupyter_node1]: Joint states are available now
[WARN] [1777659553.421857652] [behavior_recorder_jupyter_node1]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.


⚠️ Execution finished but pose not reached


False

In [59]:
move_to_pose_robust(cup_pen_grasp_pose)

[WARN] [1777586971.369883144] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777586971.370764910] [behavior_recorder_jupyter_node1]: Joint states are available now


True

In [58]:
move_to_pose_robust(cup_tilt_pose)

[WARN] [1777586961.197576387] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777586961.198243596] [behavior_recorder_jupyter_node1]: Joint states are available now


True

# sensors

## audio

In [ ]:
import subprocess
import os
from datetime import datetime

audio_proc = None
current_audio_file = None

def start_audio_record(trial_num, save_path="./audio"):
    global audio_proc, current_audio_file

    os.makedirs(save_path, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"trial_{trial_num}_{timestamp}.wav"
    full_path = os.path.join(save_path, filename)

    cmd = [
        "arecord",
        "-D", "plughw:2,0",
        "-f", "S16_LE",
        "-r", "16000",
        "-c", "1",
        full_path
    ]

    print(f"🎤 Start recording → {full_path}")

    audio_proc = subprocess.Popen(cmd)
    current_audio_file = full_path

    return full_path

def stop_audio_record():
    global audio_proc, current_audio_file

    if audio_proc is not None:
        print("⏹ Stop recording")

        audio_proc.terminate()
        audio_proc.wait()

        audio_proc = None

        # 🔥 VERY IMPORTANT: let file finalize
        time.sleep(0.2)

        return current_audio_file

    return None

In [ ]:
start_audio_record(trial_num=0)
time.sleep(2.0)
stop_audio_record()

🎤 Start recording → ./audio/trial_0_20260430_173723.wav


Recording WAVE './audio/trial_0_20260430_173723.wav' : Signed 16 bit Little Endian, Rate 16000 Hz, Mono


⏹ Stop recording


Aborted by signal Terminated...


## image

In [60]:
latest_image = None
capture_flag = False

from sensor_msgs.msg import Image
from cv_bridge import CvBridge

bridge = CvBridge()

def image_callback(msg):
    global latest_image, capture_flag

    if capture_flag:
        latest_image = bridge.imgmsg_to_cv2(msg, desired_encoding='bgr8')
        capture_flag = False

In [77]:
node.create_subscription(
    Image,
    "/camera/camera/color/image_raw",
    image_callback,
    10
)

In [ ]:
import cv2
import os
from datetime import datetime

import cv2
import os
from datetime import datetime
import time

def capture_image(trial_num, save_path="./image"):
    global capture_flag, latest_image

    os.makedirs(save_path, exist_ok=True)

    capture_flag = True

    # wait for image
    timeout = 2.0
    t0 = time.time()

    while capture_flag and (time.time() - t0 < timeout):
        time.sleep(0.05)

    if latest_image is None:
        print("⚠️ No image captured")
        return None

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"trial_{trial_num}_{timestamp}.png"
    full_path = os.path.join(save_path, filename)

    cv2.imwrite(full_path, latest_image)

    print(f"📸 Saved image: {full_path}")

    return full_path


In [91]:
# capture pen images
sync_state()
capture_image("pen_ref5", save_path="./image")

📸 Saved image: ./image/trial_pen_ref5_20260430_184525.png


'./image/trial_pen_ref5_20260430_184525.png'

# behaviors

## motions

In [20]:

def lift(delta_z=0.30):
    pose = get_pose()
    pose.position.z += delta_z
    move_to_pose(pose, speed=0.3)

In [28]:
import numpy as np
import time

ARM_JOINT_NAMES = [
    "joint_1",
    "joint_2",
    "joint_3",
    "joint_4",
    "joint_5",
    "joint_6",
]

def get_arm_joints():
    joint_dict = dict(zip(joint_state.name, joint_state.position))
    return [joint_dict[name] for name in ARM_JOINT_NAMES]

def reached_joint(target_joints, tol=0.05):
    current = get_arm_joints()
    return np.linalg.norm(np.array(current) - np.array(target_joints)) < tol

def sync_state(wait=0.3):
    time.sleep(wait)
    spin_some(wait)


def rotate_wrist(delta, speed=0.5, max_trial=3):
    joint_idx = moveit2.joint_names.index("joint_6")

    # --- sync once ---
    sync_state()

    joints = get_arm_joints()
    target = joints.copy()
    target[joint_idx] += delta

    # --- save speed ---
    prev_vel = moveit2.max_velocity
    prev_acc = moveit2.max_acceleration

    moveit2.max_velocity = speed
    moveit2.max_acceleration = speed

    # --- plan ONCE ---
    plan = moveit2.plan(joint_positions=target)

    if plan is None:
        print("❌ Planning failed")
        return False

    # --- execute multiple times ---
    for trial in range(max_trial):
        print(f"Execute trial {trial+1}")

        moveit2.execute(plan)
        wait_until_stable()

        # --- relaxed verification ---
        if reached_joint(target, tol=0.05):   # looser tolerance
            print("✅ Success")
            break

    else:
        print("⚠️ Not perfectly reached, but continuing")

    # --- restore speed ---
    moveit2.max_velocity = prev_vel
    moveit2.max_acceleration = prev_acc

    return True


In [50]:
close_gripper()
lift()

Sending gripper command: 0.8
Gripper done


[WARN] [1777586613.509640465] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777586613.510378285] [behavior_recorder_jupyter_node1]: Joint states are available now


In [66]:
lift(0.2)

[WARN] [1777587521.185640475] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777587521.186344421] [behavior_recorder_jupyter_node1]: Joint states are available now


In [64]:
move_to_pose_robust(cup_tilt_pose)

[WARN] [1777587494.131544303] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777587494.132177987] [behavior_recorder_jupyter_node1]: Joint states are available now


True

In [65]:
move_to_pose_robust(cup_pen_grasp_pose)

[WARN] [1777587512.512541844] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777587512.513185934] [behavior_recorder_jupyter_node1]: Joint states are available now


True

In [ ]:
move_to_pose(above_mircophone_pose)
for i in range(2):
    rotate_wrist(-1.57*2, speed=2.0)
    rotate_wrist(1.57*2, speed=2.0)
    # rotate_wrist_robust(-1.57*2, speed=2.0)
    # rotate_wrist(-1.57*2, speed=2.0)

## 1. shake bottle

In [29]:
def shake_bottle(home_pose, repeat_time=3, audio_save_path="/tmp"):
    move_to_pose_robust(home_pose)

    for r in range(repeat_time):
        print(f"\n=== Trial {r} ===")
        
        start_audio_record(trial_num=r, save_path=audio_save_path)
        time.sleep(0.3)  # small buffer so mic is ready

        rotate_wrist(1.57*2, speed=2.0)

        # small tail capture
        time.sleep(0.3)
        stop_audio_record()

        move_to_pose_robust(home_pose)

    # move_to_pose_robust(home_pose)

    return True

In [43]:
shake_bottle(home_pose=above_mircophone_pose, 
             repeat_time=1,
             audio_save_path="./audio/bottle_empty/")

[WARN] [1777586158.312579606] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777586158.313242694] [behavior_recorder_jupyter_node1]: Joint states are available now



=== Trial 0 ===
🎤 Start recording → ./audio/bottle_empty/trial_0_20260430_175601.wav


Recording WAVE './audio/bottle_empty/trial_0_20260430_175601.wav' : Signed 16 bit Little Endian, Rate 16000 Hz, Mono
[WARN] [1777586162.655344240] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777586162.655952710] [behavior_recorder_jupyter_node1]: Joint states are available now


Execute trial 1
Execute trial 2


[WARN] [1777586166.232779922] [behavior_recorder_jupyter_node1]: Action 'execute_trajectory' was unsuccessful: STATUS_ABORTED.


✅ Success
⏹ Stop recording


Aborted by signal Terminated...
[WARN] [1777586168.487562787] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777586168.488256606] [behavior_recorder_jupyter_node1]: Joint states are available now


True

## 2. tilt cup

In [82]:
close_gripper()

Sending gripper command: 0.8
Gripper done


In [83]:
move_to_pose_robust(cup_pre_tilt_pose)

[WARN] [1777588525.317552137] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777588525.318293938] [behavior_recorder_jupyter_node1]: Joint states are available now


True

In [74]:
def tilt_cup(image_path="./image", repeat_time=3):
    print("=== Tilt Cup Behavior ===")

    move_to_pose_robust(cup_pre_tilt_pose)

    saved_files = []

    for r in range(repeat_time):
        print(f"\n--- Trial {r} ---")

        move_to_pose_robust(cup_tilt_pose)

        time.sleep(1.0)  # let motion settle

        img_file = capture_image(trial_num=r, save_path=image_path)

        if img_file is not None:
            saved_files.append(img_file)

        move_to_pose_robust(cup_pre_tilt_pose)

    return saved_files

In [79]:
tilt_cup(repeat_time=9, image_path="./image")

=== Tilt Cup Behavior ===


[WARN] [1777588151.505567756] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777588151.506366409] [behavior_recorder_jupyter_node1]: Joint states are available now


⚠️ Execution finished but pose not reached

--- Trial 0 ---


[WARN] [1777588159.147602700] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777588159.148423004] [behavior_recorder_jupyter_node1]: Joint states are available now


📸 Saved image: ./image/trial_0_20260430_182929.png


...


--- Trial 8 ---


[WARN] [1777588304.496614518] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777588304.497465693] [behavior_recorder_jupyter_node1]: Joint states are available now


📸 Saved image: ./image/trial_8_20260430_183154.png


[WARN] [1777588315.155635258] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777588315.156341090] [behavior_recorder_jupyter_node1]: Joint states are available now


['./image/trial_0_20260430_182929.png',
 './image/trial_1_20260430_182946.png',
 './image/trial_2_20260430_183004.png',
 './image/trial_3_20260430_183022.png',
 './image/trial_4_20260430_183041.png',
 './image/trial_5_20260430_183059.png',
 './image/trial_6_20260430_183117.png',
 './image/trial_7_20260430_183136.png',
 './image/trial_8_20260430_183154.png']

In [84]:
move_to_pose_robust(cup_pen_grasp_pose)

[WARN] [1777588532.739515757] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777588532.740159709] [behavior_recorder_jupyter_node1]: Joint states are available now


True

In [85]:
open_gripper()

Sending gripper command: 0.0
Gripper done


## reset actions

In [39]:
move_to_pose_robust(grey_bottle_grasp_pose)

[WARN] [1777585531.231575319] [behavior_recorder_jupyter_node1]: Joint states are not available yet!
[INFO] [1777585531.232235823] [behavior_recorder_jupyter_node1]: Joint states are available now


True

In [44]:
open_gripper()

Sending gripper command: 0.0
Gripper done


In [41]:
close_gripper()

Sending gripper command: 0.8
Gripper done


In [ ]:
move_to_pose(above_mircophone_pose)

# shut down

In [ ]:
node.destroy_node()
rclpy.shutdown()

print("Shutdown complete")